#BPI2017 Process Predictor - Complete version

## Dataset Preprocessing following Weytjens and De Weerdt.
+ From .xes to DataFrame

In [6]:
from pm4py.objects.log.importer.xes import importer as xes_importer
from pm4py.objects.conversion.log import converter
import pandas as pd
import numpy as np

print("Importing BPIC 2017 Event Log...")
xes_file = "dataset/BPIC2017/BPI Challenge 2017.xes"
log = xes_importer.apply(xes_file)
df = converter.apply(log, variant=converter.Variants.TO_DATA_FRAME)
print("Log converted into DataFrame.")

Importing BPIC 2017 Event Log...


parsing log, completed traces ::   0%|          | 0/31509 [00:00<?, ?it/s]

Log converted into DataFrame.


---------------------------------------------------------------
+ Time Stamps converted to UTC Timezone format
+ Removing of duplicated rows (0)
+ Removing Cases ending after January 2017 because they are outliners (filtering on start date isn’t needed since all cases begin in 2016 and the focus is on eliminating cases that finish later (which are likely incomplete or outliers).
---------------------------------------------------------------

In [31]:
# Convert the 'time:timestamp' column to datetime with UTC timezone
df["time:timestamp"] = pd.to_datetime(df["time:timestamp"], utc=True)
print("The 'time:timestamp' column has been converted to datetime (UTC).")

# Remove duplicate rows from the DataFrame
df = df.drop_duplicates()
print("Duplicates removed.")

# Define a function to filter out cases that end after a given date.
def filter_cases_ending_before(dataset, end_date):
    case_ends_df = dataset.groupby("case:concept:name")["time:timestamp"].max().reset_index()
    # Remove timezone information before converting to period
    case_ends_df["time:timestamp"] = case_ends_df["time:timestamp"].dt.tz_localize(None)
    case_ends_df['date'] = case_ends_df["time:timestamp"].dt.to_period('M')
    
    print("\nCase filtering based on start and end dates range:")
    print("Minimum start date: No constraints")
    print("Maximum end date:", case_ends_df['date'].max(), "(1 Feb. 2017 Excluded - Possible outliners cases)")
    
    # Identify cases that end on or before the specified date.
    valid_cases = case_ends_df[case_ends_df['date'].astype(str) <= end_date]["case:concept:name"].values
    
    # Return only the cases that satisfy the condition.
    filtered_dataset = dataset[dataset["case:concept:name"].isin(valid_cases)]
    return filtered_dataset

# For BPIC 2017, filter out cases ending after January 2017.
end_date = "2017-01"
df_filtered = filter_cases_ending_before(df, end_date)
num_cases = df_filtered["case:concept:name"].nunique()
print("\nNumber of cases after filtering for cases ending before {}: {}".format(end_date, num_cases))

The 'time:timestamp' column has been converted to datetime (UTC).
Duplicates removed.

Case filtering based on start and end dates range:
Minimum start date: No constraints
Maximum end date: 2017-02 (1 Feb. 2017 Excluded - Possible outliners cases)

Number of cases after filtering for cases ending before 2017-01: 31497


---------------------------------------------------------------
+ Removal of cases with a total process time > 47.81
---------------------------------------------------------------

In [28]:
MAX_DAYS = 47.81

def limited_duration(dataset, max_duration):
    # Compute for each case the minimum (start) and maximum (end) timestamp
    agg_dict = {"time:timestamp": ["min", "max"]}
    duration_df = dataset.groupby("case:concept:name").agg(agg_dict).reset_index()
    
    # Calculate the duration of each case in days
    duration_df["duration"] = (
        (duration_df[("time:timestamp", "max")] - duration_df[("time:timestamp", "min")])
        .dt.total_seconds() / (24 * 60 * 60)
    )
    
    # Condition 1: Keep only cases with duration <= max_duration (a tiny tolerance is added)
    condition_1 = duration_df["duration"] <= max_duration * 1.00000000001
    cases_retained = duration_df.loc[condition_1, "case:concept:name"].values
    dataset = dataset[dataset["case:concept:name"].isin(cases_retained)].reset_index(drop=True)
    
    # Condition 2: Define the latest allowed start time as the maximum timestamp in the dataset minus max_duration
    latest_start = dataset["time:timestamp"].max() - pd.Timedelta(max_duration, unit='D')
    
    # Remove cases whose starting time (the minimum timestamp per case) is after latest_start
    case_starts = dataset.groupby("case:concept:name")["time:timestamp"].min().reset_index()
    condition_2 = case_starts["time:timestamp"] <= latest_start
    cases_retained_2 = case_starts.loc[condition_2, "case:concept:name"].values
    dataset = dataset[dataset["case:concept:name"].isin(cases_retained_2)].reset_index(drop=True)
    
    return dataset, latest_start

# Apply the limited_duration function to the filtered dataset
df_limited, latest_start = limited_duration(df_filtered, MAX_DAYS)
num_cases_limited = df_limited["case:concept:name"].nunique()
print("Number of cases after filtering by maximum duration:", num_cases_limited)
print("Latest allowed case start timestamp:", latest_start)
max_timestamp = df_filtered["time:timestamp"].max()
print("Maximum timestamp in dataset:", max_timestamp)

Number of cases after filtering by maximum duration: 29306
Latest allowed case start timestamp: 2016-12-14 21:20:41.113000+00:00
Maximum timestamp in dataset: 2017-01-31 16:47:05.113000+00:00


---------------------------------------------------------------
Target Columns Construction:

+ Next activity (probability)
+ Starting from activity A predict the time needed for B 
+ Starting from activity A predict the time needed to complete the case.
---------------------------------------------------------------

In [ ]:
# Se non è già presente, crea la colonna "activity"
df_limited["activity"] = df_limited["concept:name"] + "_" + df_limited["lifecycle:transition"]

# 1. NextActivity: la prossima attività per ogni evento (all'interno dello stesso caso)
df_limited["NextActivity"] = df_limited.groupby("case:concept:name")["activity"].shift(-1)

# 2. RemainingTimeToNextActivity: differenza in giorni tra il timestamp dell'evento successivo e quello corrente
df_limited["RemainingTimeToNextActivity"] = df_limited.groupby("case:concept:name")["time:timestamp"].shift(-1) - df_limited["time:timestamp"]
df_limited["RemainingTimeToNextActivity"] = df_limited["RemainingTimeToNextActivity"].dt.total_seconds() / (24 * 60 * 60)

# 3. RemainingTimeToEnd: differenza in giorni tra il timestamp dell'ultimo evento del caso e quello corrente
df_limited["RemainingTimeToEnd"] = df_limited.groupby("case:concept:name")["time:timestamp"].transform("max") - df_limited["time:timestamp"]
df_limited["RemainingTimeToEnd"] = df_limited["RemainingTimeToEnd"].dt.total_seconds() / (24 * 60 * 60)

print("Target columns created:")
print("NextActivity, RemainingTimeToNextActivity, RemainingTimeToEnd")
df_limited

---------------------------------------------------------------
+ Splitting the dataset into training (80%) and test (20%sets by applying strict time separation, including correction of targets for invalid prefixes. 
+ Final export of the pre-processed dataset in the desired format (CSV).
---------------------------------------------------------------

In [ ]:
def train_test_split(df, test_len, latest_start, targets):
    """
    Splits the dataset into training and test sets using strict temporal splitting and debiasing.

    Args:
        df (pd.DataFrame): The preprocessed event log (with target columns) as a Pandas DataFrame.
        test_len (float): The fraction of cases to be used as the test set (e.g., 0.2 for 20%).
        latest_start (Timestamp): The latest allowed start time for a case (for debiasing).
        targets (list): List of target column names (e.g., ["NextActivity", "RemainingTimeToNextActivity", "RemainingTimeToEnd"])
                        for which the targets should be corrected in the test set.
                        
    Returns:
        df_train (pd.DataFrame): The training set.
        df_test (pd.DataFrame): The test set.
    """
    # For each case, get the start time (minimum timestamp)
    case_start_times = df.groupby("case:concept:name")["time:timestamp"].min()
    sorted_case_ids = case_start_times.sort_values().index
    # Determine the index at which the test set starts (e.g., last 20% of cases)
    first_test_case_index = int(len(sorted_case_ids) * (1 - test_len))
    first_test_start_time = case_start_times.loc[sorted_case_ids[first_test_case_index]]
    
    # Test set: include cases that end on or after the first test start time.
    case_end_times = df.groupby("case:concept:name")["time:timestamp"].max()
    test_case_ids = case_end_times[case_end_times >= first_test_start_time].index
    df_test_all = df[df["case:concept:name"].isin(test_case_ids)].copy().reset_index(drop=True)
    
    # In the test set, drop events that occur after the debiasing threshold (latest_start)
    df_test = df_test_all[df_test_all["time:timestamp"] <= latest_start].copy().reset_index(drop=True)
    
    # For test set events that occur before the first test start time, set the target columns to NaN
    for target in targets:
        df_test.loc[df_test["time:timestamp"] < first_test_start_time, target] = np.nan
    
    # Training set: include cases that end before the first test start time.
    train_case_ids = case_end_times[case_end_times < first_test_start_time].index
    df_train = df[df["case:concept:name"].isin(train_case_ids)].copy().reset_index(drop=True)
    
    return df_train, df_test

# Define the target columns that we created earlier.
target_columns = ["NextActivity", "RemainingTimeToNextActivity", "RemainingTimeToEnd"]

# Set the test set share (for example, 20% of the cases).
TEST_LEN_SHARE = 0.2

# Split the dataset (df_limited) into training and test sets.
df_train, df_test = train_test_split(df_limited, TEST_LEN_SHARE, latest_start, target_columns)

print("Number of cases in training set:", df_train["case:concept:name"].nunique())
print("Number of cases in test set:", df_test["case:concept:name"].nunique())

# Export the final training and test sets to CSV files.
df_train.to_csv("bpic2017_train.csv", index=False)
df_test.to_csv("bpic2017_test.csv", index=False)
print("Training and test sets have been exported as CSV files.")